In [ ]:
# yappu-fsm — single-cell Colab runner
# 1) Upload yappu-fsm.zip to /content (Files panel) or set REPO_URL to a git URL.
REPO_URL = ""          # e.g. "https://github.com/<you>/yappu-fsm.git" (optional)
import os, subprocess, sys, zipfile
os.chdir("/content")
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "yappu-fsm"], check=False)
elif os.path.exists("yappu-fsm.zip"):
    zipfile.ZipFile("yappu-fsm.zip").extractall("/content")
assert os.path.isdir("yappu-fsm"), "Upload yappu-fsm.zip or set REPO_URL"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "yappu-fsm[hf,dev]"], check=True)
sys.path.insert(0, "/content/yappu-fsm")

# 2) Tests (Tamil prosody, decoder properties, HF generate(), clinical linker, MCC, server)
subprocess.run([sys.executable, "-m", "pytest", "-q", "yappu-fsm/tests"], check=False)

# 3) Prosody: validate a Nerisai Venba
from yappu_fsm import validate_nerisai_venba
poem = """வண்ண நிலவொளி வானில் தவழ்ந்திட
எண்ணற்ற விண்மீன்கள் ஏற்றிடும் - விண்ணில்
இருளினை நீக்கும் இனிய ஒளியால்
மருளினை மாற்றும் மதி."""
print("\n".join(validate_nerisai_venba(poem).lines()))

# 4) Clinical linker + MCC evaluation on the bundled gold set
from yappu_fsm import data_path
from yappu_fsm.clinical import ClinicalLinker, load_lexicon, evaluate
linker = ClinicalLinker(load_lexicon(data_path("icd10cm_example_lexicon.json")))
for f in linker.process("Pt has lt knee osteoarthritus. Denies SOB, but has uncontrolled HTN. R/O CAP."):
    print(f.raw_mention, "->", f.entity, f.assertion, f.laterality, f.codes, f.status)
print({k: v for k, v in evaluate(linker, data_path("gold_assertions.jsonl")).items() if k != "errors"})

# 5) Constrained decoding through model.generate() (offline tiny model; swap in any HF causal LM)
from yappu_fsm.demo import run_demo
run_demo(seed=0)

# 6) OpenAI-compatible /v1/chat/completions endpoint (background thread)
import threading, json, urllib.request
from yappu_fsm.server import serve
srv = serve(port=8000); threading.Thread(target=srv.serve_forever, daemon=True).start()
req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps({"model": "yappu-icd-linker",
                     "messages": [{"role": "user", "content": "Mild NPDR OD, family history of HTN."}]}).encode(),
    headers={"Content-Type": "application/json"})
print(json.loads(urllib.request.urlopen(req).read())["choices"][0]["message"]["content"])
